In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data_dir = "/content/drive/MyDrive/Glaucoma_pytorch"

In [ ]:
import os
print(os.listdir(data_dir))

['val', 'train', 'test']


In [ ]:
!pip install timm scikit-learn --quiet

In [ ]:
import os
import torch
import random
import numpy as np
import torch.nn as nn
import torch.optim as optim
import timm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from tqdm import tqdm

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
train_dataset = datasets.ImageFolder(os.path.join(data_dir,"train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(data_dir,"val"), transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(data_dir,"test"), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model("convnext_tiny", pretrained=True, num_classes=2)
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

In [ ]:
class_weights = torch.tensor([1.0, 1.59]).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

In [ ]:
import os

save_path = "/content/drive/MyDrive/Glaucoma_results"
os.makedirs(save_path, exist_ok=True)

print("Models will be saved to:", save_path)

Models will be saved to: /content/drive/MyDrive/Glaucoma_results


In [ ]:
scaler = torch.cuda.amp.GradScaler()
best_auc = 0
patience = 5
counter = 0

for epoch in range(20):

    model.train()
    train_loss = 0

    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    scheduler.step()

    # Validation
    model.eval()
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:,1].cpu().numpy())

    val_auc = roc_auc_score(all_labels, all_probs)

    print(f"\nEpoch {epoch+1}")
    print("Validation AUC:", val_auc)

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(),
           os.path.join(save_path, "best_model.pth"))
        counter = 0
        print("Model Saved!")
    else:
        counter += 1

    if counter >= patience:
        print("Early Stopping Triggered")
        break

/tmp/ipython-input-2296910357.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
  0%|          | 0/539 [00:00<?, ?it/s]/tmp/ipython-input-2296910357.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 539/539 [05:08<00:00,  1.75it/s]



Epoch 1
Validation AUC: 0.6837906213783473
Model Saved!


  0%|          | 0/539 [00:00<?, ?it/s]/tmp/ipython-input-2296910357.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 539/539 [02:27<00:00,  3.66it/s]



Epoch 2
Validation AUC: 0.6978608701794906
Model Saved!


  0%|          | 0/539 [00:00<?, ?it/s]/tmp/ipython-input-2296910357.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 539/539 [02:27<00:00,  3.66it/s]


...


Epoch 18
Validation AUC: 0.9637036812372283
Model Saved!


  0%|          | 0/539 [00:00<?, ?it/s]/tmp/ipython-input-2296910357.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 539/539 [02:28<00:00,  3.64it/s]



Epoch 19
Validation AUC: 0.9659890336867452
Model Saved!


  0%|          | 0/539 [00:00<?, ?it/s]/tmp/ipython-input-2296910357.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 539/539 [02:26<00:00,  3.68it/s]



Epoch 20
Validation AUC: 0.9665455908489666
Model Saved!


In [ ]:
# Load best model
model.load_state_dict(
    torch.load(os.path.join(save_path, "best_model.pth"), map_location=device)
)
model.eval()

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

print("Classification Report:\n")
print(classification_report(all_labels, all_preds, digits=4))

print("\nTest AUC:", roc_auc_score(all_labels, all_probs))

print("\nConfusion Matrix:\n", confusion_matrix(all_labels, all_preds))

Classification Report:

              precision    recall  f1-score   support

           0     0.9142    0.9287    0.9213      1766
           1     0.8846    0.8625    0.8734      1120

    accuracy                         0.9030      2886
   macro avg     0.8994    0.8956    0.8974      2886
weighted avg     0.9027    0.9030    0.9027      2886


Test AUC: 0.9617517392007766

Confusion Matrix:
 [[1640  126]
 [ 154  966]]


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

# Convert lists to numpy arrays
all_labels_np = np.array(all_labels)
all_probs_np = np.array(all_probs)

thresholds = np.arange(0.3, 0.7, 0.01)

best_sensitivity = 0
best_threshold = 0.5

print("Threshold | Sensitivity | Specificity")

for t in thresholds:
    preds_t = (all_probs_np >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(all_labels_np, preds_t).ravel()

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    print(f"{t:.2f}      | {sensitivity:.4f}      | {specificity:.4f}")

    # choose threshold maximizing sensitivity but keeping specificity ≥ 0.88
    if specificity >= 0.88 and sensitivity > best_sensitivity:
        best_sensitivity = sensitivity
        best_threshold = t

print("\nBest Threshold (Spec ≥ 0.88):", best_threshold)
print("Best Sensitivity:", best_sensitivity)

Threshold | Sensitivity | Specificity
0.30      | 0.9411      | 0.8012
0.31      | 0.9348      | 0.8143
0.32      | 0.9259      | 0.8233
0.33      | 0.9241      | 0.8313
0.34      | 0.9214      | 0.8403
0.35      | 0.9152      | 0.8477
0.36      | 0.9116      | 0.8545
0.37      | 0.9062      | 0.8635
0.38      | 0.9062      | 0.8715
0.39      | 0.9027      | 0.8822
0.40      | 0.8982      | 0.8862
0.41      | 0.8938      | 0.8947
0.42      | 0.8893      | 0.8992
0.43      | 0.8848      | 0.9037
0.44      | 0.8812      | 0.9071
0.45      | 0.8786      | 0.9105
0.46      | 0.8741      | 0.9156
0.47      | 0.8732      | 0.9196
0.48      | 0.8714      | 0.9230
0.49      | 0.8643      | 0.9253
0.50      | 0.8625      | 0.9287
0.51      | 0.8598      | 0.9366
0.52      | 0.8589      | 0.9388
0.53      | 0.8536      | 0.9422
0.54      | 0.8509      | 0.9422
0.55      | 0.8464      | 0.9439
0.56      | 0.8438      | 0.9462
0.57      | 0.8393      | 0.9473
0.58      | 0.8375      | 0.9485
0.59 

In [ ]:
best_t = 0.39
preds_best = (all_probs_np >= best_t).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report at Threshold 0.39:\n")
print(classification_report(all_labels_np, preds_best, digits=4))

print("Confusion Matrix:\n", confusion_matrix(all_labels_np, preds_best))

Classification Report at Threshold 0.39:

              precision    recall  f1-score   support

           0     0.9346    0.8822    0.9077      1766
           1     0.8294    0.9027    0.8645      1120

    accuracy                         0.8902      2886
   macro avg     0.8820    0.8924    0.8861      2886
weighted avg     0.8938    0.8902    0.8909      2886

Confusion Matrix:
 [[1558  208]
 [ 109 1011]]
